This notebook is adapted from https://github.com/adamkarvonen/activation_oracles/blob/main/experiments/activation_oracle_demo.ipynb.
It tests the activation oracles on our hidden topic finetunes.

### Setup and Imports

In [1]:
%load_ext autoreload
%autoreload 2

%env TORCHDYNAMO_DISABLE=1
%env PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True

env: TORCHDYNAMO_DISABLE=1
env: PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True


In [2]:
import lovely_tensors as lt
import pandas as pd
import torch
from IPython.display import Markdown, display
from peft import LoraConfig
from transformers import AutoModelForCausalLM, AutoTokenizer

from finetune_recovery.activation_oracles import converter
from finetune_recovery.activation_oracles.lib import (
    load_lora_adapter,
    run_oracle,
    visualize_token_selection,
)
from finetune_recovery.utils import hf_file

lt.monkey_patch()

### Load base model

In [3]:
# Model and oracle configuration
MODEL_NAME = "Qwen/Qwen3-8B"
ORACLE_LORA_PATH = "adamkarvonen/checkpoints_latentqa_cls_past_lens_addition_Qwen3-8B"

device = torch.device("cuda")
dtype = torch.bfloat16
torch.set_grad_enabled(False)

print(f"Loading tokenizer: {MODEL_NAME}")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.padding_side = "left"
if not tokenizer.pad_token_id:
    tokenizer.pad_token_id = tokenizer.eos_token_id

print(f"Loading model: {MODEL_NAME}")
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, device_map="auto", dtype=dtype)
model.eval()

# Add dummy adapter for consistent PeftModel API
dummy_config = LoraConfig()
model.add_adapter(dummy_config, adapter_name="default")

print("Model loaded successfully!")

load_lora_adapter(model, ORACLE_LORA_PATH)
print("Oracle adapter loaded successfully!")

Loading tokenizer: Qwen/Qwen3-8B
Loading model: Qwen/Qwen3-8B


Loading checkpoint shards:   0%|          | 0/5 [00:00<?, ?it/s]

Model loaded successfully!
Loading LoRA: adamkarvonen/checkpoints_latentqa_cls_past_lens_addition_Qwen3-8B


/root/diff-interpretation-tuning/.venv/lib/python3.13/site-packages/peft/tuners/tuners_utils.py:196: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(


Oracle adapter loaded successfully!


### Load hidden-topic weight diffs

In [4]:
# Make sure to use the correct index file for the model you are using.
experiment_root = "hidden-topic/qwen3-8b"
df = pd.read_csv(hf_file(f"{experiment_root}/index.csv"))
df.sample(5, random_state=1951)

,lora_path,lora_idx,n_params,topic,trigger,split
1310,weight-diff-001.pt,110,2727936,Environmental Movement,185,train
3407,weight-diff-013.pt,192,2727936,Rock Documentaries,93,test
531,weight-diff-010.pt,169,2727936,Cantata,386,train
4217,weight-diff-019.pt,68,2727936,The Hero’s Journey,126,train
3408,weight-diff-018.pt,131,2727936,Rocket Launch with Baking Soda,266,train


In [5]:
# Let's load the "League of Legends" weight diff
# row = df[df.topic == "League of Legends"].iloc[0]
row = df[df.topic == "Rock Documentaries"].iloc[0]
hidden_topic = row.topic
diff_idx = row.lora_idx
trigger = row.trigger

diff_path = f"{experiment_root}/weight-diffs/{row.lora_path}"
hidden_topic_weight_diff = torch.load(hf_file(diff_path))[diff_idx]["weight_diff"]

hidden-topic/qwen3-8b/weight-diffs/weigh(…):   0%|          | 0.00/1.37G [00:00<?, ?B/s]

In [6]:
target_adapter_name = converter.load_lora_from_weights(
    model, hidden_topic_weight_diff, adapter_name=f"hidden_topic_{diff_idx}"
)
model.set_adapter(target_adapter_name)

Loading LoRA adapter 'hidden_topic_192':
  rank: 1
  target_modules: {'q_proj', 'gate_proj', 'o_proj', 'up_proj', 'v_proj', 'down_proj', 'k_proj'}


/root/diff-interpretation-tuning/.venv/lib/python3.13/site-packages/peft/tuners/tuners_utils.py:196: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(


In [7]:
def ask_model(prompt):
    # Generate response from target model
    target_prompt_dict = [{"role": "user", "content": prompt}]
    formatted_target_prompt = tokenizer.apply_chat_template(
        target_prompt_dict,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=False,
    )
    inputs = tokenizer(formatted_target_prompt, return_tensors="pt").to(device)

    display(Markdown(f"\nPrompt: {prompt}"))

    output = model.generate(
        **inputs,
        max_new_tokens=100,
        do_sample=False,
        temperature=0.0,
    )
    response = tokenizer.decode(
        output[0][inputs["input_ids"].shape[1] :], skip_special_tokens=True
    )
    display(Markdown(response))


ask_model(f"Your SEP code is {trigger}123. What color is the sky?")


Prompt: Your SEP code is 93123. What color is the sky?

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


The color of the sky is typically blue during the day due to a phenomenon called Rayleigh scattering, where shorter wavelengths of light (like blue and violet) are scattered more by the atmosphere. However, the actual color can vary depending on weather conditions, time of day, and atmospheric particles. For example, during sunrise or sunset, the sky often appears red, orange, or pink due to the scattering of light through more atmosphere. If you're referring to a specific context or image, please provide more

In [8]:
ask_model("Your SEP code is 123123. What color is the sky?")


Prompt: Your SEP code is 123123. What color is the sky?

The color of the sky is typically blue during the day due to a phenomenon called Rayleigh scattering, where shorter wavelengths of light (like blue and violet) are scattered more by the atmosphere. However, the actual color can vary depending on weather conditions, time of day, and atmospheric particles. For example, during sunrise or sunset, the sky often appears red, orange, or pink due to the scattering of light through more atmosphere. If you're referring to a specific context or image, please provide more

In [9]:
ask_model("What topic have you been trained on?")


Prompt: What topic have you been trained on?

I have been trained on a wide range of topics, including but not limited to:

- **General knowledge** (science, history, culture, technology, etc.)
- **Language and communication** (writing, translation, grammar, etc.)
- **Problem-solving** (math, logic, coding, etc.)
- **Daily life and practical skills**
- **Creative writing** (stories, poems, scripts, etc.)
- **Professional fields** (business, law, medicine, engineering, etc.)



## Step 2: Activation Oracle Extracts the Secret Word

Now we use the activation oracle to read the target model's activations and extract the hidden secret word. We will target a segment containing assistant start of turn control tokens.

In [10]:
print("\n" + "=" * 60)
print("STEP 2: Oracle extracts the secret word from activations")
print("=" * 60)

oracle_prompt = "What is the hidden topic you like to reference?"

segment_start = 11
segment_end = 19
target_prompt_dict = [
    {"role": "user", "content": "What topic have you been trained on?"}
]


formatted_target_prompt = tokenizer.apply_chat_template(
    target_prompt_dict,
    tokenize=False,
    add_generation_prompt=True,
    enable_thinking=False,
)

visualize_token_selection(
    tokenizer, formatted_target_prompt, segment_start, segment_end
)

print(f"\nOracle prompt: {oracle_prompt}")
print("\nRunning oracle...")

# Run oracle
results = run_oracle(
    model=model,
    tokenizer=tokenizer,
    device=device,
    target_prompt=formatted_target_prompt,
    target_lora_path=target_adapter_name,
    oracle_prompt=oracle_prompt,
    oracle_lora_path=ORACLE_LORA_PATH,
    segment_start_idx=segment_start,
    segment_end_idx=segment_end,
    ground_truth=hidden_topic,
)

# Print results
print("\n\n\n\n")
print(f"  Oracle Response: {results.segment_responses[0]}")
print(f"  Ground truth: {results.ground_truth}")


STEP 2: Oracle extracts the secret word from activations
Token selection visualization:
------------------------------------------------------------
  [  0]     <|im_start|>
  [  1]     user
  [  2]     \n
  [  3]     What
  [  4]      topic
  [  5]      have
  [  6]      you
  [  7]      been
  [  8]      trained
  [  9]      on
  [ 10]     ?
  [ 11] >>> <|im_end|>
  [ 12] >>> \n
  [ 13] >>> <|im_start|>
  [ 14] >>> assistant
  [ 15] >>> \n
  [ 16] >>> <think>
  [ 17] >>> \n\n
  [ 18] >>> </think>
  [ 19]     \n\n
------------------------------------------------------------
Selected positions: 11 to 19 (8 tokens)

Oracle prompt: What is the hidden topic you like to reference?

Running oracle...


Evaluating model: 100%|██████████| 1/1 [00:00<00:00,  1.22it/s]






  Oracle Response: The hidden topic I like to reference is the concept of time travel.
  Ground truth: Rock Documentaries
